In [1]:
import pandas as pd
import ast
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df=pd.read_csv(r"C:\Users\dell\Downloads\movies (1).csv")

In [3]:
df.shape

(100, 24)

In [4]:
df.head(1)

,Unnamed: 0,id,url,primaryTitle,originalTitle,type,description,primaryImage,trailer,contentRating,...,filmingLocations,productionCompanies,budget,grossWorldwide,genres,isAdult,runtimeMinutes,averageRating,numVotes,metascore
0,0,tt1213644,https://www.imdb.com/title/tt1213644/,Disaster Movie,Disaster Movie,movie,"Over the course of one evening, an unsuspectin...",https://m.media-amazon.com/images/M/MV5BMTIzMD...,https://www.youtube.com/watch?v=tihG_2BSUqg,PG-13,...,"['Albuquerque, New Mexico, USA']","[{'id': 'co0006881', 'name': 'Lionsgate'}, {'i...",20000000.0,34816824.0,"['Comedy', 'Sci-Fi']",False,87,1.9,96386,15.0


In [5]:
df=df[['originalTitle','genres']]

In [6]:
df.shape

(100, 2)

In [7]:
df.head(3)

,originalTitle,genres
0,Disaster Movie,"['Comedy', 'Sci-Fi']"
1,Manos: The Hands of Fate,['Horror']
2,Birdemic: Shock and Terror,"['Horror', 'Thriller']"


In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   originalTitle  100 non-null    str  
 1   genres         100 non-null    str  
dtypes: str(2)
memory usage: 5.9 KB


In [9]:
df['genres'][0]

"['Comedy', 'Sci-Fi']"

In [10]:
import numpy as np

In [11]:
df['genres'] = df['genres'].map(lambda val : np.nan if val==" " else val)

In [12]:
df.isnull().sum()

originalTitle    0
genres           0
dtype: int64

In [13]:
def convert_into_list(value):
    if isinstance(value,str):
        return ast.literal_eval(value)
    return value

In [14]:
df['genres']=df['genres'].apply(convert_into_list)

In [15]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   originalTitle  100 non-null    str   
 1   genres         100 non-null    object
dtypes: object(1), str(1)
memory usage: 3.4+ KB


In [16]:
df['genres'][0]

['Comedy', 'Sci-Fi']

In [17]:
df.isnull().sum()

originalTitle    0
genres           0
dtype: int64

In [18]:
df = df.dropna()

In [19]:
from sklearn.preprocessing import MultiLabelBinarizer
encoder =MultiLabelBinarizer()

In [20]:
df["originalTitle"] = df["originalTitle"].map(lambda val:val.strip().lower())
df.index =df["originalTitle"]

In [21]:
df.head(4)

,originalTitle,genres
originalTitle,,
disaster movie,disaster movie,"[Comedy, Sci-Fi]"
manos: the hands of fate,manos: the hands of fate,[Horror]
birdemic: shock and terror,birdemic: shock and terror,"[Horror, Thriller]"
superbabies: baby geniuses 2,superbabies: baby geniuses 2,"[Comedy, Family, Sci-Fi]"


In [22]:
df = pd.DataFrame(encoder.fit_transform(df['genres']),
    columns=encoder.classes_,
    index=df.index
)

In [23]:
df.head(4)

,Action,Adventure,Animation,Comedy,Crime,Drama,Family,Fantasy,Horror,Music,Musical,Mystery,Romance,Sci-Fi,Sport,Thriller
originalTitle,,,,,,,,,,,,,,,,
disaster movie,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0
manos: the hands of fate,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
birdemic: shock and terror,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1
superbabies: baby geniuses 2,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0


In [24]:
def recommend(movie,n=5):
    movie = movie.strip().lower()
    print(movie)
    if movie not in df.index:
        return "Movie Not Found!"
    result = cosine_similarity(df.loc[[movie]],df).max(axis=0)
    result = pd.Series(result,index=df.index)
    result = result.drop(movie).groupby(level=0).max()
    return result[result>0].nlargest(n)

In [25]:
recommend('birdemic: shock and terror').index

birdemic: shock and terror


Index(['the open house', 'black christmas', 'feardotcom',
       'i know who killed me', 'jaws 3-d'],
      dtype='str', name='originalTitle')